<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex07.2-heat-and-wave/Ex07.2_1_die_pulse.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_07.2 · 1 (Optional) — The Silicon Die After a Power Pulse (parabolic)

**Deep Learning for Engineering · Aalborg University · Part 2 · paired with L7.2 · Fundamental PDEs**

A chip has just finished a burst of work. Its silicon die is hot in a pattern
the burst left behind, and the package it sits on pulls the heat out through
its edges. How does the temperature fall, and where does it stay hot longest?
The heat equation answers it; you solve it by finite differences and by a
physics-informed network, and score both against the exact solution.

This notebook stands on its own: the slot and the panel are the other two
notebooks of the set, and neither is needed here.

| | what you do | TODO |
|---|---|---|
| **1 – 3** | the problem, its data and its physics | — |
| **4** | the exact solution, and finite differences on a mesh for an agreed 0.02 K | — |
| **5** | a PINN with the start and the edges built in, and the comparison | the trial function, the residual |
| **6** | what the comparison says | — |
| **7 – 8** | your report, and the mini project | your answers |

About five minutes on a CPU. Numbers are printed with at most two decimals.

---

## 0 · Setup

In [ ]:
# files-cell v2 ----------------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex07.2-heat-and-wave/"
ON_COLAB = "google.colab" in sys.modules
for f in FILES:
    if ON_COLAB or not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
for _m in ("course_core", "pinn_core", "problem"):
    sys.modules.pop(_m, None)
print("files ready:", ", ".join(FILES))

In [ ]:
# --- setup ---------------------------------------------------------------
import os, time
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"
from pinn_core import *                                  # noqa: F401,F403
import course_core as cc
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt
import scipy.sparse as sp, scipy.sparse.linalg as spla

cc.OUTPUT_DIR = "Ex07.2_outputs"                          # where the report is written
set_seed(88)  # fix the random start so a run repeats
print("torch", torch.__version__, " device:", DEVICE, " dtype:", torch.get_default_dtype())

In [ ]:
def middle(fn, reps=5):                                    # the middle of five timings, after a warm-up
    fn(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

---

## 1 · The problem

A 10 × 10 mm silicon die, its edges clamped to the package at 65 °C. At the
end of a power pulse it is up to 60 K hotter than the package, and nothing
heats it any more. What is the temperature at every point of the die over the
next 200 ms?

## 2 · The data

| | |
|---|---|
| die | 10 × 10 mm silicon, $L$ = 10 mm |
| thermal diffusivity | $\alpha$ = 8.8e-05 m²/s |
| edges | held at the package's 65 °C: a rise of $\theta$ = 0 |
| start | the field the pulse left: a broad hump of 40 K and a sharp one of 20 K, superposed |
| window | 0 to 200 ms |

## 3 · The physics

$$\theta_t = \alpha\,(\theta_{xx} + \theta_{yy}), \qquad \theta = 0 \text{ on the edges}, \qquad \theta(x, y, 0) = \theta_0(x, y)$$

The heat equation is **parabolic**: the temperature has a direction in time and
forgets. It needs **one** initial condition, and every feature of the start
decays - the sharper the feature, the faster. The start here is two of the
die's modes, so the exact solution is two decaying humps:

$$\theta = 40\,\sin\tfrac{\pi x}{L}\sin\tfrac{\pi y}{L}\,e^{-2\pi^2\alpha t/L^2} + 20\,\sin\tfrac{2\pi x}{L}\sin\tfrac{\pi y}{L}\,e^{-5\pi^2\alpha t/L^2}$$

The broad hump decays at 17.37 per second, the sharp one at 43.43 - two and a
half times faster - so the field changes **shape** as it cools, not only size.
The cell below draws it at four instants.

In [ ]:
L, T_END = pb.L_DIE, pb.HEAT_T_END
k_slow, k_fast = pb.heat_rates()
print(f"decay rates: broad hump {k_slow:.2f} /s, sharp hump {k_fast:.2f} /s")
g = np.linspace(0, L, 81); X, Y = np.meshgrid(g, g)
fig, axes = plt.subplots(1, 4, figsize=(13, 3.2))
for ax, t in zip(axes, (0.0, 0.02, 0.05, 0.2)):
    c = ax.contourf(X * 1e3, Y * 1e3, pb.heat_exact(X, Y, t), 24, cmap="inferno")
    ax.set_title(f"t = {t*1e3:.0f} ms, peak {pb.heat_exact(X, Y, t).max():.2f} K"); ax.set_aspect("equal")
    ax.set_xlabel("x  [mm]"); plt.colorbar(c, ax=ax, shrink=0.8)
axes[0].set_ylabel("y  [mm]")
plt.tight_layout(); plt.show()

**What you should see.** Decay rates of 17.37 and 43.43 per second, and a
field that starts lopsided - its peak about 52 K - and becomes one centred
hump as the sharp mode dies, down to about 1.2 K at 200 ms.

---

## 4 · The reference, and finite differences

**The reference** is the exact solution above: at any point and any time it is
right to the last digit, so every method is scored against it.

**Finite differences.** The die's interior nodes are the unknowns, the
five-point stencil stands in for the Laplacian, and Crank-Nicolson steps the
field in time - one sparse solve per step, with the matrix factorised once. The
cell below runs four meshes, each twice as fine in space and time as the last,
and scores each against the exact solution over the whole window. **The agreed
accuracy is 0.02 K**; the coarsest mesh that meets it is the one used from here
on.

In [ ]:
def fdm_die(n, steps):
    # n interior nodes a side; returns node coordinates and the field at every step
    h = L / (n + 1); x = np.arange(1, n + 1) * h
    X, Y = np.meshgrid(x, x)
    I1, T1 = sp.identity(n), sp.diags([1.0, -2.0, 1.0], [-1, 0, 1], shape=(n, n))
    A = pb.ALPHA * (sp.kron(I1, T1) + sp.kron(T1, I1)) / h ** 2       # the five-point Laplacian, times alpha
    dt = T_END / steps; I = sp.identity(n * n)
    lu = spla.splu((I - 0.5 * dt * A).tocsc()); B = (I + 0.5 * dt * A).tocsr()   # Crank-Nicolson
    u = pb.heat_initial(X, Y).ravel(); out = [u]
    for _ in range(steps):
        u = lu.solve(B @ u); out.append(u)
    return X, Y, np.array(out)

TARGET = 0.02                                                 # K: the agreed accuracy
for n, steps in ((9, 50), (19, 100), (39, 200), (79, 400)):
    X, Y, U = fdm_die(n, steps)
    ts = np.linspace(0, T_END, steps + 1)
    err = max(np.abs(U[k] - pb.heat_exact(X, Y, t).ravel()).max() for k, t in enumerate(ts))
    sec = middle(lambda: fdm_die(n, steps))
    print(f"  {n+2:3d} x {n+2} nodes, {steps} steps: worst error {err:.2e} K, one run {sec*1e3:.0f} ms"
          + ("   <- meets 0.02 K" if err <= TARGET else ""))
N_FDM, STEPS = 39, 200                                        # the coarsest mesh that meets it

**What you should see.** The error falling four times with every halving -
2.52e-01, 6.26e-02, 1.58e-02, 3.94e-03 K - and the 41 × 41 mesh with 200 steps
the first under 0.02 K, the whole window in a few tens of milliseconds.

---

## 5 · A physics-informed network, and the comparison

**The network** takes $(X, Y, \tau)$ - position divided by $L$, time divided by
the 200 ms window - and returns the rise. It is four layers of 32 tanh neurons,
trained on 2000 collocation points spread by Latin hypercube over the die and
the window.

**The start and the edges are built in** (L7.2's **Hard Initial Condition**
slide): the trial function

$$\hat\theta = \theta_0(x, y) + \tau\;16X(1-X)Y(1-Y)\;\Theta\,\mathcal{N}(X, Y, \tau)$$

is exactly $\theta_0$ at $\tau = 0$, whatever the network does, and exactly zero on
every edge, because $\theta_0$ is and the mask is. $\Theta$ = 60 K is the size
of the answer, so the network only learns a number of order one.

**TODO 1** is the trial function; **TODO 2** the residual. In the scaled
coordinates the heat equation reads $\hat\theta_\tau = K(\hat\theta_{XX} + \hat\theta_{YY})$
with $K = \alpha T/L^2$ = 0.18; divided by $\Theta$ it is of order one.

In [ ]:
THETA = 60.0                                                  # K: the size of the answer
class Die(torch.nn.Module):                                   # a network: layers in __init__, the computation in forward
    def __init__(self):
        super().__init__()
        self.net = MLP(n_in=3, n_hidden=32, n_layers=4)       # dense tanh network: X, Y, tau in
    def forward(self, p):
        X, Y, tau = p[:, :1], p[:, 1:2], p[:, 2:]
        # TODO 1: the trial function - the start plus tau times the mask times THETA times the network.
        #
        #   start = pb.heat_initial(X * L, Y * L)
        #   return start + tau * 16 * X * (1 - X) * Y * (1 - Y) * THETA * self.net(p)
        raise NotImplementedError("Write the trial function")

In [ ]:
K = pb.ALPHA * T_END / L ** 2                                 # 0.18: the heat equation in scaled units
# TODO 2: the residual of the heat equation, divided by THETA.
#
#   def residual(model, p):
#       th = model(p)
#       th_tau = grad(th, p)[:, 2:]                               # column 2 is tau
#       return (th_tau - K * (d2(th, p, 0) + d2(th, p, 1))) / THETA
raise NotImplementedError("Write residual(model, p)")

In [ ]:
pts = to_tensor(interior_points(2000, ((0, 1), (0, 1), (0, 1)), method="lhs", seed=1), requires_grad=True)  # collocation points, differentiable
model = Die().to(DEVICE)                                      # four layers of 32
describe(model.net, n_collocation=len(pts))                   # the parameter count and the collocation-to-parameter ratio
t0 = time.perf_counter()
# Adam first, then L-BFGS: the Part 2 training recipe
train_two_stage(model, lambda: residual(model, pts).pow(2).mean(),
                adam_steps=1500, lbfgs_steps=150, lr=2e-3, report_every=0)
train_time = time.perf_counter() - t0
print(f"trained in {train_time:.0f} s")

In [ ]:
X, Y, U = fdm_die(N_FDM, STEPS)                               # the finite-difference answer, every step
ts = np.linspace(0, T_END, STEPS + 1)
Pn = to_tensor(np.c_[X.ravel() / L, Y.ravel() / L])           # the same nodes for the network
def pinn_at(t):
    with torch.no_grad():  # no gradient tracking: evaluation only
        return to_numpy(model(torch.cat([Pn, torch.full((len(Pn), 1), t / T_END, device=Pn.device, dtype=Pn.dtype)], 1))).ravel()   # the time column on the same device as the nodes
e_fdm = np.array([np.abs(U[k] - pb.heat_exact(X, Y, t).ravel()).max() for k, t in enumerate(ts)])
e_pin = np.array([np.abs(pinn_at(t) - pb.heat_exact(X, Y, t).ravel()).max() for t in ts])
t_fdm = middle(lambda: fdm_die(N_FDM, STEPS))
t_pin = middle(lambda: [pinn_at(t) for t in ts])
print(f"{'':28s}{'worst error':>13s}{'whole window':>15s}{'training':>10s}")
print(f"  {'finite differences':26s}{e_fdm.max():11.2e} K{t_fdm*1e3:12.1f} ms{'-':>10s}")
print(f"  {'PINN':26s}{e_pin.max():11.2e} K{t_pin*1e3:12.1f} ms{train_time:8.0f} s")
fig, ax = plt.subplots(figsize=(6.4, 3.8))
ax.semilogy(ts * 1e3, e_fdm, label="finite differences, 41 x 41"); ax.semilogy(ts * 1e3, e_pin, label="PINN")
ax.axhline(TARGET, color="k", lw=0.8, ls="--", label="agreed 0.02 K")
ax.set_xlabel("t  [ms]"); ax.set_ylabel("worst error over the die  [K]"); ax.legend(frameon=False)
plt.tight_layout(); plt.savefig(cc.output_path("Ex07.2_1_die_pulse_report.png"), dpi=150); plt.show()

**What you should see.** Finite differences at 1.58e-02 K and the network at
about 1.65e-02 K - both within the agreed 0.02 K - and both errors falling as
the die cools. Finite differences take about 30 ms for the whole window and no
training. The network needs about three minutes of training, and then about
300 ms to give the same 201 instants on the same nodes.

---

## 6 · What the comparison says

* **Both meet the agreed accuracy.** The start is built into the network, so it
  is exact at $t = 0$, and the heat equation only smooths from there: an error
  made early is damped, not carried.
* **Finite differences win outright on this die.** They solve the whole window
  faster than the trained network can even be evaluated over it, and they need
  no training. For one die with one start, the network has nothing to offer.
* **The parabolic type is on the network's side.** The field gets smoother and
  simpler with time - exactly what a smooth network represents well. The wave of
  notebook 3 is the opposite case.

---

## 7 · Your report

**What these cells do.** You answer the questions below; the next cell writes `Ex07.2_1_die_pulse_report.md` with the results of section 5 and your answers, and the cell after it turns the report into a PDF for Moodle.

Each question builds part of an answer to one of L7.2's questions for the oral examination; the arrow says which. Keep each answer to a few sentences, with your numbers. Write your name in `NAME`, and each answer between the triple quotes that follow its question; an answer needs at least eight words.

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "What does 'parabolic' mean for the die, and how many initial conditions did it need? (-> L7.2 Q1)": """
""",
    "Why does the field change shape as it cools, and which of the two humps sets the late temperature? (-> L7.2 Q3)": """
""",
    "What does building the start into the trial function guarantee, and what does it cost the network? (-> L7.2 Q4)": """
""",
    "To conclude: finite differences met 0.02 K in about 30 ms. For which die problem would a trained network be worth its minutes of training? (-> L7.2, Exercise slide)": """
""",
}

In [ ]:
MIN_WORDS = 8                                               # an answer is a few sentences, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:
    rows = ["| method | worst error | whole window | training |", "|---|---|---|---|",
            f"| finite differences, 41 x 41, 200 steps | {e_fdm.max():.2e} K | {t_fdm*1e3:.1f} ms | - |",
            f"| PINN, 4 x 32 | {e_pin.max():.2e} K | {t_pin*1e3:.1f} ms | {train_time:.0f} s |"]
    out = ["# Ex_07.2 — The Silicon Die After a Power Pulse", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("Ex07.2_1_die_pulse_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figure of section 5, and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex07.2_1_die_pulse_report.md into Ex07.2_1_die_pulse_report.pdf, with any figure
# saved as Ex07.2_1_die_pulse_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "Ex07.2_1_die_pulse_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "Ex07.2_1_die_pulse_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("Ex07.2_1_die_pulse_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "Ex07.2_1_die_pulse_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass

---

## 8 · Mini project proposal

One problem grown out of this notebook, for you to take further on your own. Each
student chooses **one** mini project from the Part 2 sets and has **one month**
to solve it individually. The dates and the hand-in are given in L13.

The ground truth is **given to you**, built by the course with a classical
solver and shown to have converged; the worked example below each project is
its first case. You do not solve the physics a second way: the month goes into
the deep learning - making a network predict what the ground truth says, as
accurately and as fast as the requirements ask.

### MP7.2A · A die with a moving hot spot

**The problem.** The die of this notebook was released from a fixed initial field and cooled. A
working chip is never that quiet: its power comes from a few hot blocks that
switch on and off as the load moves between them, and silicon conducts less
as it warms, so the heat equation becomes nonlinear.

| | |
|---|---|
| die | 10 × 10 mm silicon, edges held at the package's 65 °C, 0.5 mm thick |
| blocks | three, 2 × 2 mm, centred at (2.5, 2.5), (7.5, 2.5) and (5.0, 7.5) mm |
| power | 8 W per block - 200 W/cm², a hot spot's flux - each on for 60 ms in turn, from 0 to 180 ms |
| silicon | $k = 148\,(300/T)^{1.3}$ W/m·K: 126.7 at 65 °C, 111.4 at 100 °C; heat capacity 1.68 MJ/m³·K |
| window | 0 to 200 ms, starting at 65 °C everywhere |

The die thickness, the block power and the fit for $k(T)$ are typical values,
sourced before the data is released.

**Why a plain PINN is not enough.** Each switch-on starts a sharp local transient that the network must resolve
in both space and time, the peak moves from block to block, and the
conductivity now depends on the answer. Uniform collocation spends most of its
points where nothing happens.

**What you build.** A PINN for the rise $\theta(x, y, t)$ with the power map as a known input,
the initial condition built in as in this notebook, and a sampling that follows
the power: more points near the active block and just after each switch. You
decide how to split the window, and whether training in time order helps.

**The ground truth you get.** `MP7.2A_truth.npz`, from `tools/miniprojects/ex072_truth.py`: the die
solved by cell-centred finite volumes with backward Euler in time, the
conductivity iterated to convergence at every step. Cells of 0.05 mm and steps
of 0.5 ms; the field every 2 ms, the power map, $k(T)$ and the refinement
table.

**Worked example.** 

| cells | time step | peak rise | when | one run |
|---|---|---|---|---|
| 0.2 mm | 2 ms | 33.64 K | 180 ms | 1.6 s |
| 0.1 mm | 1 ms | 33.73 K | 180 ms | 15.5 s |
| 0.05 mm | 0.5 ms | 33.79 K | 180 ms | 149.7 s |

The hottest moment is the end of the third block's pulse, at 98.79 °C. With
the conductivity held at its 65 °C value the peak is 32.08 K: silicon's $k(T)$
adds **1.66 K**, which a linear model misses.

**Requirements.**

* The peak rise within **1 K**, and its time within **2 ms**.
* The temperature within **2 K** everywhere, at every recorded instant.
* Say whether your network sees the 1.66 K that $k(T)$ adds.
* Once trained, the whole 200 ms on the 0.05 mm grid in under **1 s** - against 150 s for the finite volumes.

**What you hand in.** The error in space and time against the ground truth, the peak and when it
happens, how much of the error sits at the switching instants, and the times.
Say what a designer could trust your network for.